In [1]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("df_bert.csv")

df.rename(columns={"readmission_30day": "labels"}, inplace=True)

In [3]:
df

,hadm_id,text,labels
0,22595853,5491 V1582 30981 29680 496 07070 \nName: ___...,0
1,22841357,5491 3051 V08 5715 496 2761 \nName: ___ ...,1
2,29079034,nan 5715 29680 496 V462 V4986 \nName: ___ ...,1
3,25742920,5491 78791 3051 V08 496 2761 \nName: ___ ...,0
4,23052089,nan Z8546 E785 R296 R441 F0280 \nName: ___ ...,0
...,...,...,...
317478,24755486,3E03305 F17210 D472 E876 C8580 Z5111 \nName: ...,0
317479,29734428,3E0436Z 02HV33Z 0HBHXZZ 0HBJXZZ 0HR7X74 I9581 ...,0
317480,25744818,05HY33Z 0WPF0JZ 0J980ZZ R197 E60 B954 E876 F41...,0
317481,26071774,8841 8891 3051 2724 4019 43811 34590 \nName: ...,0


In [4]:
df.rename(columns = {"readmission_30day": "labels"})

,hadm_id,text,labels
0,22595853,5491 V1582 30981 29680 496 07070 \nName: ___...,0
1,22841357,5491 3051 V08 5715 496 2761 \nName: ___ ...,1
2,29079034,nan 5715 29680 496 V462 V4986 \nName: ___ ...,1
3,25742920,5491 78791 3051 V08 496 2761 \nName: ___ ...,0
4,23052089,nan Z8546 E785 R296 R441 F0280 \nName: ___ ...,0
...,...,...,...
317478,24755486,3E03305 F17210 D472 E876 C8580 Z5111 \nName: ...,0
317479,29734428,3E0436Z 02HV33Z 0HBHXZZ 0HBJXZZ 0HR7X74 I9581 ...,0
317480,25744818,05HY33Z 0WPF0JZ 0J980ZZ R197 E60 B954 E876 F41...,0
317481,26071774,8841 8891 3051 2724 4019 43811 34590 \nName: ...,0


In [5]:
def preprocess1(x):
    y = re.sub(r'\[(.*?)\]', '', x)              # Removes content in brackets [Text]
    y = re.sub(r'[0-9]+\.', '', y)               # Removes numbered list markers (1. 2. 3.)
    y = re.sub(r'dr\.', 'doctor', y)             # Standardizes 'dr.' to 'doctor'
    y = re.sub(r'm\.d\.', 'md', y)               # Standardizes 'm.d.' to 'md'
    y = re.sub(r'--|__|==|_', '', y)             # Removes separator characters
    y = re.sub(r'name:', '', y)                  # Removes patient name
    y = re.sub(r'unit no:', '', y)               # Removes patient unit no
    y = re.sub(r'admission date:', '', y)        # Removes 'admission date:' header
    y = re.sub(r'discharge date:', '', y)        # Removes 'discharge date:' header
    y = re.sub(r'date of birth:', '', y)         # Removes date of birth line
    y = re.sub(r'attending: .*?\n', '', y)       # Removes 'attending:' line
    return y


def preprocessing(df):
    
    df['text'] = df['text'].fillna(' ')
    df['text'] = df['text'].str.replace('\n', ' ')
    df['text'] = df['text'].str.replace('\r', ' ')
    df['text'] = df['text'].apply(str.strip)
    df['text'] = df['text'].str.lower()
    df['text'] = df['text'].apply(lambda x: preprocess1(x)) # Apply cleaning using preprocess1 func

    # Collect chunks in a list instead of DataFrame.append
    chunks = []

    for i in tqdm(range(len(df))):
        words = df.text.iloc[i].split()
        n = len(words) // 128   # number of full 128-word chunks

        # Add full chunks
        for j in range(n):
            chunks.append({
                'hadm_id': df.hadm_id.iloc[i],
                'text': ' '.join(words[j*128:(j+1)*128]),
                'labels': df.labels.iloc[i]
            })

        # Add leftover chunk if > 10 words
        leftover = len(words) % 128
        if leftover > 10:
            chunks.append({
                'hadm_id': df.hadm_id.iloc[i],
                'text': ' '.join(words[-leftover:]),
                'labels': df.labels.iloc[i]
            })

    # Convert once at the end
    return pd.DataFrame(chunks)

In [6]:
# Apply preprocessing to all data
df_processed = preprocessing(df)

print("Preprocessed data shape:", df_processed.shape)
print("Label distribution:\n", df_processed["labels"].value_counts())


100%|█████████████████████████████████| 317483/317483 [02:00<00:00, 2629.29it/s]


Preprocessed data shape: (3897648, 3)
Label distribution:
 labels
0    2973155
1     924493
Name: count, dtype: int64


In [7]:
# get unique admissions for Train/Val/Test Split
unique_admissions = df_processed.groupby('hadm_id').agg({
    'labels': 'first'
}).reset_index()

# Split: 70% train, 30% temp for further splitting
train_ids, temp_ids = train_test_split(
    unique_admissions['hadm_id'], 
    test_size=0.3, 
    random_state=42,
    stratify=unique_admissions['labels']
)

# Split temp into val and test
temp_data = unique_admissions[unique_admissions['hadm_id'].isin(temp_ids)]
val_ids, test_ids = train_test_split(
    temp_data['hadm_id'], 
    test_size=0.5, 
    random_state=42,
    stratify=temp_data['labels']
)



In [13]:
# Filtering preprocessed data to get only training data
df_final_bert = df_processed[df_processed['hadm_id'].isin(train_ids)].reset_index(drop=True)
df_val_bert = df_processed[df_processed['hadm_id'].isin(val_ids)].reset_index(drop=True)
df_test_bert = df_processed[df_processed['hadm_id'].isin(test_ids)].reset_index(drop=True)

# get statistics
print(f"Training set: {len(df_final_bert)} samples from {len(train_ids)} unique admissions")
print(f"Validation set: {len(df_val_bert)} samples from {len(val_ids)} unique admissions")
print(f"Test set: {len(df_test_bert)} samples from {len(test_ids)} unique admissions")
print(f"\nTraining label distribution:\n{df_final_bert['labels'].value_counts()}")
print(f"\nValidation label distribution:\n{df_val_bert['labels'].value_counts()}")
print(f"\nTest label distribution:\n{df_test_bert['labels'].value_counts()}")



Training set: 2729276 samples from 222238 unique admissions
Validation set: 585577 samples from 47622 unique admissions
Test set: 582795 samples from 47623 unique admissions

Training label distribution:
labels
0    2080695
1     648581
Name: count, dtype: int64

Validation label distribution:
labels
0    447361
1    138216
Name: count, dtype: int64

Test label distribution:
labels
0    445099
1    137696
Name: count, dtype: int64


In [12]:
# Saving only the training data to prevent data leakage since data is already huge 
df_final_bert.to_csv("df_final_bert.csv", index=False)

# # Optional if needed
# df_val_bert.to_csv("df_val_bert.csv", index=False)
# df_test_bert.to_csv("df_test_bert.csv", index=False)